In [4]:
# These are the libraries you may use for this assignment.
# If you would like to use any other libraries, email james@uwaterloo.ca for permission first.
# Generative AI tools (including coding assistants) are not permitted for this assignment.

from IPython.display import display, Math, Latex

import pandas as pd
import numpy as np
import numpy_financial as npf
import yfinance as yf
import matplotlib.pyplot as plt
import random
from datetime import datetime

## CFM 101 — Group Assignment 2026
### Robo-Advising Challenge: Two Clients, Two Strategies

**Team Number:** 12
**Team Member Names (three or four):**  

Enter the names of all participating members below (one name per line):

Daniel Lei
Turki 
Ruoxi
Grace

**Submission:** Friday, November 27, 2026, at 9:00 AM (Eastern), via Dropbox.  
**Submit this completed Jupyter notebook only.**

Your team must construct **both** portfolios from the supplied `Tickers.csv` file:

- **Client A — Growth:** $\$1,000,000$ CAD; seek the highest expected return.
- **Client B — Low Risk:** $\$750,000$ CAD; seek the lowest portfolio volatility.

Both clients require 10–20 eligible stocks, at most 15% in any stock and at most 40% in any sector. Read the full assignment handout for all eligibility, fee, currency, and competition rules. Do not hardcode the selected stocks. Check that tickers are valid. The supplied ticker universe will permit feasible portfolios.

**No generative AI may be used in preparing this assignment.**

## Data preparation and shared functions

Read `Tickers.csv`, check ticker eligibility, retrieve the data needed for your strategies, and create any functions you will reuse for both clients. Use closing prices from **November 27, 2026** for the purchase calculations. All portfolio values and weights must be calculated in CAD, with purchase fees paid from the client budget.

In [3]:
all_tickers = read_tickers(TICKER_FILE)
print(f"{len(all_tickers)} unique tickers read from {TICKER_FILE}")

purchase_ts = pd.Timestamp(PURCHASE_DATE)
end_exclusive = (purchase_ts + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

close_raw, volume_raw = download_panel(all_tickers, HIST_START, end_exclusive)
aux_close, _ = download_panel([FX_TICKER, MARKET_TICKER], HIST_START, end_exclusive)
if FX_TICKER not in aux_close.columns or aux_close[FX_TICKER].dropna().empty:
    raise RuntimeError("Could not download the USD/CAD exchange rate (CAD=X) - cannot value USD stocks in CAD.") 

NameError: name 'read_tickers' is not defined

## Client A — Growth

Construct a portfolio intended to achieve the **highest expected return**. Use your selected stocks and portfolio weights to create a DataFrame named `Portfolio_Final_A` with index **1 through n** and columns (in this order): `Ticker`, `Price`, `Currency`, `Shares`, `Value`, `Weight`. `Price` is in the stock’s quoted currency, `Value` is in CAD, and `Weight` is a fraction summing to 1.

| Signal | Why it points toward higher return | Weight |
|---|---|---|
| **Beta** (Chapter 15) | In a rising market, a beta-1.5 stock tends to move about 1.5× the market. CAPM: more market risk earns more expected return. | 40% |
| **12-1 month momentum** | Past-year winners (skipping the latest month, which tends to bounce back) have tended to keep winning (Jegadeesh & Titman, 1993). Research evidence, not from our slides. | 40% |
| **Revenue growth** (Yahoo) | Our client wants *growth*; fast-growing sales support a company's story. Yahoo's data is patchy, so it gets less weight and **a missing value counts as neutral**. | 20% |

**Selection:** top 12 by score, at most 4 per sector. **Weighting:** proportional to score (shifted so the lowest of the 12 is still positive),
then the 15% / 40% caps are applied and any excess is spread over the uncapped stocks.
**What we deliberately did *not* use:** dividends — the competition ignores dividends, so they earn nothing in this contest.

### Client A — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.
Client A aims for the maximum return and is willing to take high risks. As expected returns cannot be observed, I have measured each stock against three factors and aggregated them using their z-scores (their distance from the mean of the entire set). Beta will account for 40%: stock with beta greater than 1 will move even more upwards than the movement of the market, and hence, the theory suggests that investors get higher compensation for market risk according to CAPM. Momentum over 12-1 month period will get 40%. 20% for revenue growth that corresponds to the growth strategy, although it is given less weight since the data for Yahoo is unreliable and the absence of a figure means a neutral score. I have purchased the top 12 based on their scores, where each sector is allowed a maximum of 4 stocks so that no single sector will be dominant. This was done intentionally since the contest gives recognition solely based on the best performing stock over five days, and with fewer stocks, there is greater variation in performance. Higher scores receive larger allocations with the stock limit of 15% and the sector limit of 40%. 

Client A is likely high-scoring (higher beta, growth, strong past year): so stocks like 
AMZN, SHOP.TO, QCOM, LLY, PYPL, AXP, CAT, BA, plus possibly C, BAC, TXN and ACN. BB.TO is very volatile and could score high or low depending on its momentum. The 4-per-sector limit will push the list to spread across tech, financials, industrials and healthcare.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.
If the market falls between Nov 30 and Dec 4, high-beta stocks will fall more than the market, so the beta tilt works against us. Momentum can reverse suddenly, and over only five days luck matters more than any signal. Up to 40% of the portfolio can sit in one sector, so a sector-specific shock would hurt. US stocks also lose value in CAD if the Canadian dollar strengthens.

## Client B — Low Risk

Construct a portfolio intended to achieve the **lowest risk**, defined as the standard deviation of daily portfolio returns in CAD. Create `Portfolio_Final_B` using the same DataFrame column names, order, and index convention as for Client A.

### Client B — Financial justification

**Why does this portfolio make financial sense?** Explain how and why you select and weight stocks.

**What evidence supports your strategy?** Discuss relevant statistics and the equal-weight comparison below.

**What could go wrong?** Identify at least one important limitation or situation in which this strategy may perform poorly.

## Equal-weight comparisons

For **each client**, create an equal-weighted comparison portfolio holding **exactly the same stocks** as that client’s chosen portfolio, with starting weight `1/n` per stock. Over the same clearly identified historical period, calculate **historical return** and **volatility** for the chosen portfolio and its equal-weighted counterpart. Use consistent CAD valuations and weighting assumptions.

Display one concise comparison table per client, and interpret the results in the financial justifications above. No additional graph or out-of-sample test is required.

## Programmatic validation

Use Python (`assert` statements or a simple function are fine) to display whether **both** final portfolios pass each of the following checks:

- Selected tickers are valid and eligible.
- Number of stocks is between 10 and 20.
- No individual stock weight exceeds 15%.
- No single sector has more than 40% of portfolio value.
- Portfolio weights sum to 100%.
- Total stock purchases plus transaction fees do not exceed the applicable client budget.

Your code should run with the TAs’ ticker file without manually changing individual stock selections.

## Final portfolio outputs

Display `Portfolio_Final_A` and `Portfolio_Final_B`, along with the budget reconciliation (purchases plus fees) for each. Generate `Stocks_Group_XX_A.csv` and `Stocks_Group_XX_B.csv` (replace `XX` with your team number). Each CSV must contain **only** the columns `Ticker` and `Shares`. The notebook must **generate** these CSV files when run, but you submit **only the notebook** to Dropbox.

## Contribution and AI Declaration

**The following team members made a meaningful contribution to this assignment:**

Enter the names of all contributing members below (one name per line):

**To the best of our knowledge, no generative AI was used in preparing this submission.**

List only members who made a meaningful contribution. Contact the instructor before the deadline if someone should be omitted or if the team cannot make the declaration.